In [ ]:
import os
import time
import mne
import numpy as np
import pandas as pd

import bsl
from bsl import StreamPlayer, datasets
# from bsl.externals import pylsl  # distributed version of pylsl
from bsl.triggers import TriggerDef

import pylsl

import pickle

import math
import matplotlib
import matplotlib.pyplot as plt
from pythonosc.udp_client import SimpleUDPClient

In [ ]:
from pylsl import resolve_streams

streams = resolve_streams()
for s in streams:
    print(s.name(), s.type(), s.channel_count(), s.nominal_srate())

### OSC Client Intialization

In [ ]:
# OSC client initialization
ip = "127.0.0.1"
port = 7000
client = SimpleUDPClient(ip, port)

### Load Pretrained Model

In [ ]:
# Load your pretrained model and scaler
with open("model.pkl", "rb") as f_m:
    model = pickle.load(f_m)
with open("scaler.pkl", "rb") as f_s:
    scaler = pickle.load(f_s)
with open("pca.pkl", "rb") as f_p:
    pca = pickle.load(f_p)

## Analyse Signal

In [ ]:
from pylsl import resolve_streams, StreamInlet
from collections import deque
import numpy as np
# from scipy.signal import butter, sosfilt  # was used for EEG
import mne
import time

# ── Constants ────────────────────────────────────────────────────────────────
# EEG_FS = 250
FNIRS_FS = 75.0
WINDOW_S = 10
# EEG_WINDOW_SAMPLES = int(EEG_FS * WINDOW_S)    # 2500 samples
FNIRS_WINDOW_SAMPLES = int(FNIRS_FS * WINDOW_S) # 750 samples
UPDATE_EVERY_S = 0.5
N_FNIRS_CHANNELS = 28

HBO_COLS = list(range(0, N_FNIRS_CHANNELS, 2))
HBR_COLS = list(range(1, N_FNIRS_CHANNELS, 2))

# ── MNE info objects (TODO: update with real values from stream) ──────────────
# eeg_info = mne.create_info(sfreq=EEG_FS, ch_names=['E1'], ch_types=['eeg'])

ch_names = [
    'Rx1 - Tx1 O2Hb', 'Rx1 - Tx1 HHb',
    'Rx1 - Tx3 O2Hb', 'Rx1 - Tx3 HHb',
    'Rx2 - Tx1 O2Hb', 'Rx2 - Tx1 HHb',
    'Rx2 - Tx3 O2Hb', 'Rx2 - Tx3 HHb',
    'Rx3 - Tx4 O2Hb', 'Rx3 - Tx4 HHb',
    'Rx3 - Tx5 O2Hb', 'Rx3 - Tx5 HHb',
    'Rx8 - Tx9 O2Hb', 'Rx8 - Tx9 HHb',
    'Rx8 - Tx10 O2Hb', 'Rx8 - Tx10 HHb',
    'Rx5 - Tx6 O2Hb', 'Rx5 - Tx6 HHb',
    'Rx5 - Tx8 O2Hb', 'Rx5 - Tx8 HHb',
    'Rx6 - Tx6 O2Hb', 'Rx6 - Tx6 HHb',
    'Rx6 - Tx8 O2Hb', 'Rx6 - Tx8 HHb',
    'Rx4 - Tx2 O2Hb', 'Rx4 - Tx2 HHb',
    'Rx7 - Tx7 O2Hb', 'Rx7 - Tx7 HHb',
]
fnirs_info = mne.create_info(ch_names=ch_names, sfreq=FNIRS_FS, ch_types=['fnirs_cw_amplitude'] * 28)

# ── Baseline normalization (TODO: compute from real baseline recording) ───────
ref_mean_score = 0
ref_std_score = 1  # avoid div by zero

# ── EEG bandpass filter 1-30Hz ───────────────────────────────────────────────
# def make_bandpass(lowcut, highcut, fs, order=4):
#     nyq = fs / 2
#     sos = butter(order, [lowcut / nyq, highcut / nyq], btype='band', output='sos')
#     return sos
#
# eeg_sos = make_bandpass(1, 30, EEG_FS)

# ── LSL inlets ───────────────────────────────────────────────────────────────
streams = resolve_streams()
# eeg_stream = [s for s in streams if s.type() == 'EEG'][0]
fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
# eeg_inlet = StreamInlet(eeg_stream)
fnirs_inlet = StreamInlet(fnirs_stream)

# ── Ring buffers ─────────────────────────────────────────────────────────────
# eeg_buffer = deque(maxlen=EEG_WINDOW_SAMPLES)
fnirs_buffer = deque(maxlen=FNIRS_WINDOW_SAMPLES)

# ── Main loop ─────────────────────────────────────────────────────────────────
last_process_time = time.time()

while True:
    # Ingest both streams
    # eeg_samples, _ = eeg_inlet.pull_chunk(timeout=0.1)
    # if eeg_samples:
    #     eeg_buffer.extend(eeg_samples)

    fnirs_samples, _ = fnirs_inlet.pull_chunk(timeout=0.0)  # non-blocking second call
    if fnirs_samples:
        fnirs_buffer.extend(fnirs_samples)

    # Wait for fNIRS buffer to fill on startup
    if len(fnirs_buffer) < FNIRS_WINDOW_SAMPLES:
        print(f"Buffering... fNIRS {len(fnirs_buffer)}/{FNIRS_WINDOW_SAMPLES}")
        continue

    # Rate limit processing
    now = time.time()
    if now - last_process_time < UPDATE_EVERY_S:
        continue
    last_process_time = now

    # ── EEG features (disabled) ──────────────────────────────────────────────
    # eeg_data = np.array(eeg_buffer)             # (2500, n_eeg_channels)
    # eeg_data = np.nan_to_num(eeg_data)
    #
    # # TODO: update column selection once real channel layout is known
    # eeg_channel = eeg_data[:, 0]                # (2500,) single channel for now
    #
    # eeg_filt = sosfilt(eeg_sos, eeg_channel)    # bandpass 1-30Hz
    #
    # # Alpha power (8-12Hz) via Welch
    # from scipy.signal import welch
    # freqs, psd = welch(eeg_filt, fs=EEG_FS, nperseg=EEG_FS // 2)
    # alpha_mask = (freqs >= 8) & (freqs <= 12)
    # alpha_score = psd[alpha_mask].mean()
    # alpha_norm = (alpha_score - ref_mean_score) / ref_std_score

    # ── fNIRS features ────────────────────────────────────────────────────────
    fnirs_data = np.array(fnirs_buffer)         # (750, 28)
    fnirs_data = np.nan_to_num(fnirs_data)

    hbo = fnirs_data[:, HBO_COLS]               # (750, 14)
    hbr = fnirs_data[:, HBR_COLS]               # (750, 14)

    hbo_mean = hbo.mean(axis=0)                 # (14,)
    hbr_mean = hbr.mean(axis=0)                 # (14,)

    t = np.arange(FNIRS_WINDOW_SAMPLES) / FNIRS_FS
    hbo_slope = np.polyfit(t, hbo, 1)[0]       # (14,)
    hbr_slope = np.polyfit(t, hbr, 1)[0]       # (14,)

    # ── Fusion & prediction ───────────────────────────────────────────────────
    features = np.concatenate([hbo_mean, hbr_mean, hbo_slope, hbr_slope])  # (56,)  # EEG (alpha_norm) disabled
    # 112 raw features
    features = np.array(features).reshape(1, -1)

    # Scale
    features_scaled = scaler.transform(features)

    # PCA
    features_pca = pca.transform(features_scaled)

    # Predict
    prediction = model.predict(features_pca)

    print("Raw features:", features.shape)
    print("Scaled features:", features_scaled.shape)
    print("PCA features:", features_pca.shape)
    print("Prediction:", prediction)

## Claude Code of EEG fNIRS ML prediction using MLE

In [ ]:
from pylsl import resolve_streams, StreamInlet
from collections import deque
import numpy as np
from scipy.signal import butter, sosfilt
import mne
import pickle
import time

# ── Constants ────────────────────────────────────────────────────────────────
EEG_FS = 250
FNIRS_FS = 75.0
WINDOW_S = 10
EEG_WINDOW_SAMPLES = int(EEG_FS * WINDOW_S)    # 2500 samples
FNIRS_WINDOW_SAMPLES = int(FNIRS_FS * WINDOW_S) # 750 samples
UPDATE_EVERY_S = 0.5
N_FNIRS_CHANNELS = 28
EPS = 1e-12

HBO_COLS = list(range(0, N_FNIRS_CHANNELS, 2))
HBR_COLS = list(range(1, N_FNIRS_CHANNELS, 2))

# ── MNE info objects (TODO: update with real values from stream) ──────────────
# eeg_info = mne.create_info(sfreq=EEG_FS, ch_names=['E1'], ch_types=['eeg'])

ch_names = [
    'Rx1 - Tx1 O2Hb', 'Rx1 - Tx1 HHb',
    'Rx1 - Tx3 O2Hb', 'Rx1 - Tx3 HHb',
    'Rx2 - Tx1 O2Hb', 'Rx2 - Tx1 HHb',
    'Rx2 - Tx3 O2Hb', 'Rx2 - Tx3 HHb',
    'Rx3 - Tx4 O2Hb', 'Rx3 - Tx4 HHb',
    'Rx3 - Tx5 O2Hb', 'Rx3 - Tx5 HHb',
    'Rx8 - Tx9 O2Hb', 'Rx8 - Tx9 HHb',
    'Rx8 - Tx10 O2Hb', 'Rx8 - Tx10 HHb',
    'Rx5 - Tx6 O2Hb', 'Rx5 - Tx6 HHb',
    'Rx5 - Tx8 O2Hb', 'Rx5 - Tx8 HHb',
    'Rx6 - Tx6 O2Hb', 'Rx6 - Tx6 HHb',
    'Rx6 - Tx8 O2Hb', 'Rx6 - Tx8 HHb',
    'Rx4 - Tx2 O2Hb', 'Rx4 - Tx2 HHb',
    'Rx7 - Tx7 O2Hb', 'Rx7 - Tx7 HHb',
]
fnirs_info = mne.create_info(ch_names=ch_names, sfreq=FNIRS_FS, ch_types=['fnirs_cw_amplitude'] * 28)

# ── Baseline normalization (TODO: compute from real baseline recording) ───────
ref_mean_score = 0
ref_std_score = 1  # avoid div by zero

# ── EEG bandpass filter 1-30Hz ───────────────────────────────────────────────
def make_bandpass(lowcut, highcut, fs, order=4):
    nyq = fs / 2
    sos = butter(order, [lowcut / nyq, highcut / nyq], btype='band', output='sos')
    return sos

eeg_sos = make_bandpass(1, 30, EEG_FS)

# ── Load trained models ────────────────────────────────────────────────────
with open("scaler_fnirs.pkl", "rb") as f:
    scaler = pickle.load(f)
with open("pca_fnirs.pkl", "rb") as f:
    pca = pickle.load(f)
with open("model_fnirs.pkl", "rb") as f:
    model = pickle.load(f)

with open("scaler_eeg.pkl", "rb") as f:
    eeg_scaler = pickle.load(f)
with open("pca_eeg.pkl", "rb") as f:
    eeg_pca = pickle.load(f)
with open("model_eeg.pkl", "rb") as f:
    eeg_model = pickle.load(f)

# ── LSL inlets ───────────────────────────────────────────────────────────────
streams = resolve_streams()
eeg_stream = [s for s in streams if s.type() == 'EEG'][0]
fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
eeg_inlet = StreamInlet(eeg_stream)
fnirs_inlet = StreamInlet(fnirs_stream)

# ── Ring buffers ─────────────────────────────────────────────────────────────
eeg_buffer = deque(maxlen=EEG_WINDOW_SAMPLES)
fnirs_buffer = deque(maxlen=FNIRS_WINDOW_SAMPLES)

# ── Main loop ─────────────────────────────────────────────────────────────────
last_process_time = time.time()

while True:
    # Ingest both streams
    eeg_samples, _ = eeg_inlet.pull_chunk(timeout=0.1)
    if eeg_samples:
        eeg_buffer.extend(eeg_samples)

    fnirs_samples, _ = fnirs_inlet.pull_chunk(timeout=0.0)  # non-blocking second call
    if fnirs_samples:
        fnirs_buffer.extend(fnirs_samples)

    # Wait for both buffers to fill on startup
    if len(eeg_buffer) < EEG_WINDOW_SAMPLES or len(fnirs_buffer) < FNIRS_WINDOW_SAMPLES:
        print(f"Buffering... EEG {len(eeg_buffer)}/{EEG_WINDOW_SAMPLES}  fNIRS {len(fnirs_buffer)}/{FNIRS_WINDOW_SAMPLES}")
        continue

    # Rate limit processing
    now = time.time()
    if now - last_process_time < UPDATE_EVERY_S:
        continue
    last_process_time = now

    # ── EEG features ──────────────────────────────────────────────────────────
    eeg_data = np.array(eeg_buffer)             # (2500, n_eeg_channels)
    eeg_data = np.nan_to_num(eeg_data)

    # TODO: update column selection once real channel layout is known
    eeg_channel = eeg_data[:, 0]                # (2500,) single channel for now

    eeg_filt = sosfilt(eeg_sos, eeg_channel)    # bandpass 1-30Hz

    # Alpha power (8-12Hz) via Welch
    from scipy.signal import welch
    freqs, psd = welch(eeg_filt, fs=EEG_FS, nperseg=EEG_FS // 2)
    alpha_mask = (freqs >= 8) & (freqs <= 12)
    alpha_score = psd[alpha_mask].mean()
    alpha_norm = (alpha_score - ref_mean_score) / ref_std_score

    # ── fNIRS features ────────────────────────────────────────────────────────
    fnirs_data = np.array(fnirs_buffer)         # (750, 28)
    fnirs_data = np.nan_to_num(fnirs_data)

    hbo = fnirs_data[:, HBO_COLS]               # (750, 14)
    hbr = fnirs_data[:, HBR_COLS]               # (750, 14)

    hbo_mean = hbo.mean(axis=0)                 # (14,)
    hbr_mean = hbr.mean(axis=0)                 # (14,)

    t = np.arange(FNIRS_WINDOW_SAMPLES) / FNIRS_FS
    hbo_slope = np.polyfit(t, hbo, 1)[0]       # (14,)
    hbr_slope = np.polyfit(t, hbr, 1)[0]       # (14,)

    # ── Fusion & prediction ───────────────────────────────────────────────────
    fnirs_features = np.concatenate([hbo_mean, hbr_mean, hbo_slope, hbr_slope])  # (56,)
    fnirs_features = np.array(fnirs_features).reshape(1, -1)

    eeg_features = np.array([[alpha_norm]])     # (1, 1)  # TODO: expand once eeg feature set is finalized

    # Scale
    fnirs_scaled = scaler.transform(fnirs_features)
    eeg_scaled = eeg_scaler.transform(eeg_features)

    # PCA
    fnirs_pca_feat = pca.transform(fnirs_scaled)
    eeg_pca_feat = eeg_pca.transform(eeg_scaled)

    # Predict probabilities from each modality
    proba_fnirs = model.predict_proba(fnirs_pca_feat)
    proba_eeg = eeg_model.predict_proba(eeg_pca_feat)

    # ── MLE ensemble (product-of-experts) ───────────────────────────────────
    # Align class ordering between the two models (they might differ if one
    # modality's training data is missing a class)
    classes = np.array(sorted(set(model.classes_) | set(eeg_model.classes_)))
    class_index = {c: i for i, c in enumerate(classes)}

    def expand_proba(proba, model_classes):
        out = np.full((proba.shape[0], len(classes)), EPS)
        for i, c in enumerate(model_classes):
            out[:, class_index[c]] = proba[:, i]
        return out

    proba_fnirs_full = expand_proba(proba_fnirs, model.classes_)
    proba_eeg_full = expand_proba(proba_eeg, eeg_model.classes_)

    pred_fnirs = classes[proba_fnirs_full.argmax(axis=1)[0]]
    pred_eeg = classes[proba_eeg_full.argmax(axis=1)[0]]
    agree = pred_fnirs == pred_eeg

    # Maximum-likelihood combination: multiply the two likelihood vectors
    # (equivalently, sum their logs), then renormalize
    log_combined = np.log(proba_fnirs_full + EPS) + np.log(proba_eeg_full + EPS)
    log_combined -= log_combined.max(axis=1, keepdims=True)  # stability
    combined = np.exp(log_combined)
    combined /= combined.sum(axis=1, keepdims=True)

    prediction = classes[combined.argmax(axis=1)[0]]
    confidence = combined.max(axis=1)[0]

    print("fNIRS features:", fnirs_features.shape)
    print("EEG features:", eeg_features.shape)
    print("fNIRS prediction:", pred_fnirs)
    print("EEG prediction:", pred_eeg)
    print("Agree:", agree)
    print("Prediction:", prediction, f"(confidence={confidence:.3f})")